In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
import joblib

## 1. Data Loading and Preprocessing

In [ ]:
df = pd.read_csv('creditcard_sampled.csv')

# Assuming 'Class' is the target variable for credit card fraud datasets
# If your target column is named differently, please update 'Class' to your target column name.
if 'Class' in df.columns:
    df = df.rename(columns={'Class': 'target'})
else:
    print("Warning: 'Class' column not found, assuming 'target' already exists or will be created.")

for col in df.select_dtypes(include='object').columns:
    df[col] = LabelEncoder().fit_transform(df[col])

X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Data loaded, preprocessed, and split into training and testing sets.")


Data loaded, preprocessed, and split into training and testing sets.


## 2. Model Definition

In [9]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(eval_metric="logloss", use_label_encoder=False, random_state=42)
}

print("Models defined.")


Models defined.


## 3. Model Training and Evaluation

In [10]:
best_model = None
best_acc = 0

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    print(f"{name}: Accuracy = {acc:.4f}")

    if acc > best_acc:
        best_acc = acc
        best_model = model

print(f"\nTraining and evaluation complete. Best accuracy found: {best_acc:.4f}")


c:\Users\kandr\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:470: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression: Accuracy = 0.9990
Decision Tree: Accuracy = 0.9992
Random Forest: Accuracy = 0.9996


c:\Users\kandr\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:52:56] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost: Accuracy = 0.9995

Training and evaluation complete. Best accuracy found: 0.9996


In [12]:
app_py_content = """from flask import Flask, render_template, request
import joblib
import numpy as np

app = Flask(__name__)

try:
    model = joblib.load("credit_card_model.pkl")
except Exception as e:
    model = None
    print("Model load failed:", e)

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/predict", methods=["POST"])
def predict():
    try:
        data = [float(x) for x in request.form.values()]
    except Exception:
        return render_template("index.html", prediction="Invalid input")

    if model is None:
        return render_template("index.html", prediction="Model not available")

    pred = model.predict([data])[0]

    result = "Fraudulent Transaction" if pred == 1 else "Legitimate Transaction"

    return render_template("index.html", prediction=result)

if __name__ == "__main__":
    app.run(debug=True)
"""

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_py_content)

print("Updated app.py in project root with corrected prediction logic.")

Updated app.py in project root with corrected prediction logic.


## 4. Save Best Model

In [13]:
joblib.dump(best_model, "credit_card_model.pkl")
print(f"Best model saved to credit_card_model.pkl with accuracy: {best_acc:.4f}")


Best model saved to credit_card_model.pkl with accuracy: 0.9996
